# R Master v9 Fix5 · TRUE Body Mesh Compare

**Fix4 结果作废：它按顶点数误选了 Lap 的 Face_Mesh。**

Fix5 精确选择 Lap 的 `Body_Mesh`，用两套骨架建立人体坐标并排除手臂权重区域；继续 No-Render，避免 iPhone Colab 在 Eevee 渲染阶段断线。

不改源模型，不转权重，不烘 Rest Pose，不导 VRM。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import shutil, subprocess, json, zipfile, tarfile, os

ROOT=Path("/content/drive/MyDrive/R_Master")
REF=ROOT/"reference"
CACHE=ROOT/"cache"
OUT=ROOT/"v9_body_compare"/"fix5_latest"
OUT.mkdir(parents=True,exist_ok=True)
LOCAL=Path("/content/r_master_v9_fix5")
LOCAL.mkdir(parents=True,exist_ok=True)

mona_candidates=[
    ROOT/"v7c_refine/latest/R_Master_v7c_REFINED_PREVIEW.blend",
    ROOT/"v8c_head_beauty/latest/R_Master_v8c_HEAD_BEAUTY_PREVIEW.blend",
]
MONA=next((p for p in mona_candidates if p.exists()),None)
if MONA is None:
    raise FileNotFoundError("Mona source not found")

LAPZIP=REF/"Lapine_Ver.1.11_A.zip"
if not LAPZIP.exists():
    raise FileNotFoundError(LAPZIP)

EXTRACTED=REF/"extracted"
EXTRACTED.mkdir(exist_ok=True)
LAPFBX=EXTRACTED/"Lapine_BaseBody.fbx"

if not LAPFBX.exists() or LAPFBX.stat().st_size<10*1024*1024:
    TMP=LOCAL/"lap_extract"
    if TMP.exists():
        shutil.rmtree(TMP)
    TMP.mkdir()
    with zipfile.ZipFile(LAPZIP,"r") as z:
        unity=[n for n in z.namelist() if n.lower().endswith("lapine.unitypackage")]
        if not unity:
            raise RuntimeError("Lapine.unitypackage missing")
        upkg=TMP/"Lapine.unitypackage"
        upkg.write_bytes(z.read(unity[0]))
    with tarfile.open(upkg,"r:*") as t:
        paths={}
        assets={}
        for m in t.getmembers():
            parts=m.name.split("/")
            if len(parts)!=2:
                continue
            d,name=parts
            if name=="pathname":
                paths[d]=t.extractfile(m).read().decode("utf-8","ignore").strip()
            elif name=="asset":
                assets[d]=m
        target=next((d for d,p in paths.items() if p.endswith("Lapine_BaseBody.fbx")),None)
        if target is None:
            raise RuntimeError("Lapine_BaseBody.fbx missing")
        LAPFBX.write_bytes(t.extractfile(assets[target]).read())

BLENDER_VERSION="4.4.3"
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
BLENDER=BDIR/"blender"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if not DRIVE_ARCHIVE.exists() or DRIVE_ARCHIVE.stat().st_size<100*1024*1024:
    raise RuntimeError("Blender cache missing")

shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
if not BLENDER.exists():
    if BDIR.exists():
        shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)

if not BLENDER.exists():
    raise FileNotFoundError(BLENDER)

print("✓ Mona:",MONA)
print("✓ Lap FBX:",LAPFBX,round(LAPFBX.stat().st_size/1024/1024,1),"MiB")
print("✓ Blender ready")
\n

In [ ]:
REPORT=OUT/"R_Master_v9_Fix5_true_body_profiles.json"
SCRIPT=LOCAL/"v9_fix5_extract.py"
SCRIPT.write_text(r'''
import bpy,sys,os,json,math
from mathutils import Vector

argv=sys.argv[sys.argv.index("--")+1:]
def arg(k):
    return argv[argv.index(k)+1]

mona=arg("--mona")
lap=arg("--lap")
out=arg("--out")
os.makedirs(out,exist_ok=True)

def bone_any(rig,names):
    for n in names:
        b=rig.data.bones.get(n)
        if b:
            return b
    for b in rig.data.bones:
        low=b.name.lower().replace(" ","").replace("_","").replace(".","")
        for n in names:
            key=n.lower().replace(" ","").replace("_","").replace(".","")
            if low==key or low.endswith(key) or key in low:
                return b
    return None

def world_head(rig,b):
    return rig.matrix_world @ b.head_local

def rig_frame(rig):
    hips=bone_any(rig,["Hips"])
    head=bone_any(rig,["Head"])
    lu=bone_any(rig,["LeftUpperLeg","thigh.L","upper_leg.L","L_UpperLeg"])
    ru=bone_any(rig,["RightUpperLeg","thigh.R","upper_leg.R","R_UpperLeg"])
    lf=bone_any(rig,["LeftFoot","foot.L","L_Foot"])
    rf=bone_any(rig,["RightFoot","foot.R","R_Foot"])
    neck=bone_any(rig,["Neck"])
    chest=bone_any(rig,["Chest","UpperChest","Spine2","spine.002"])
    spine=bone_any(rig,["Spine","spine.001"])

    required=[hips,head,lu,ru,lf,rf]
    if not all(required):
        raise RuntimeError("required rig landmarks missing: "+rig.name)

    H=world_head(rig,hips)
    HD=world_head(rig,head)
    L=world_head(rig,lu)
    R=world_head(rig,ru)
    FL=world_head(rig,lf)
    FR=world_head(rig,rf)

    up=(HD-H).normalized()
    right=(R-L).normalized()
    fwd=right.cross(up).normalized()
    right=up.cross(fwd).normalized()

    foot=(FL+FR)*0.5
    stature=(HD-foot).dot(up)
    if stature<=0:
        raise RuntimeError("bad skeletal stature: "+rig.name)

    def zn(b):
        if b is None:
            return None
        return ((world_head(rig,b)-foot).dot(up))/stature

    return {
        "origin":foot,
        "up":up,
        "right":right,
        "fwd":fwd,
        "stature":stature,
        "z":{
            "foot":0.0,
            "hips":zn(hips),
            "spine":zn(spine),
            "chest":zn(chest),
            "neck":zn(neck),
            "head":zn(head)
        }
    }

def local_points(obj,frame,exclude_arms=True):
    M=obj.matrix_world
    names={g.index:g.name.lower() for g in obj.vertex_groups}
    arm_tokens=("upperarm","upper_arm","lowerarm","lower_arm","forearm","shoulder",
                "elbow","hand","thumb","index","middle","ring","little","finger",
                "wrist","arm_twist")
    out=[]
    for v in obj.data.vertices:
        if exclude_arms and v.groups:
            arm_weight=0.0
            for ge in v.groups:
                n=names.get(ge.group,"")
                if any(t in n for t in arm_tokens):
                    arm_weight=max(arm_weight,ge.weight)
            if arm_weight>0.20:
                continue

        p=M @ v.co
        d=p-frame["origin"]
        out.append((
            d.dot(frame["right"])/frame["stature"],
            d.dot(frame["fwd"])/frame["stature"],
            d.dot(frame["up"])/frame["stature"]
        ))
    return out

def profile(points,angle_deg,zmin,zmax,bins=240):
    a=math.radians(angle_deg)
    ca,sa=math.cos(a),math.sin(a)
    rows=[]
    for i in range(bins):
        z0=zmin+(zmax-zmin)*i/bins
        z1=zmin+(zmax-zmin)*(i+1)/bins
        vals=[x*ca+y*sa for x,y,z in points if z0<=z<z1]
        if vals:
            rows.append({
                "z":(z0+z1)*0.5,
                "min":min(vals),
                "max":max(vals),
                "width":max(vals)-min(vals)
            })
    return rows

def slab(points,z,band=.012):
    pts=[p for p in points if abs(p[2]-z)<=band]
    if not pts:
        return None
    xs=[p[0] for p in pts]
    ys=[p[1] for p in pts]
    return {
        "width":max(xs)-min(xs),
        "depth":max(ys)-min(ys),
        "count":len(pts)
    }

# Mona
bpy.ops.wm.open_mainfile(filepath=mona)
mona_body=bpy.data.objects.get("R2_Mona_Main") or bpy.data.objects.get("Mona_Main")
mona_rig=bpy.data.objects.get("R_Master_Align_v2_PREVIEW") or bpy.data.objects.get("Mona_Armature")
if not mona_body or not mona_rig:
    raise RuntimeError("Mona body/rig missing")
MF=rig_frame(mona_rig)
MP=local_points(mona_body,MF,True)

# Lap
before=set(bpy.data.objects)
bpy.ops.import_scene.fbx(filepath=lap)
imports=[o for o in bpy.data.objects if o not in before]
lap_meshes=[o for o in imports if o.type=="MESH"]
lap_rigs=[o for o in imports if o.type=="ARMATURE"]
audit=[{"name":o.name,"verts":len(o.data.vertices)} for o in sorted(lap_meshes,key=lambda o:o.name)]

# Exact body selection. Never choose by vertex count.
lap_body=next((o for o in lap_meshes if o.name=="Body_Mesh" or o.name.startswith("Body_Mesh.")),None)
if lap_body is None:
    raise RuntimeError("Exact Lap Body_Mesh not found. Meshes="+str(audit))

lap_rig=next((r for r in lap_rigs if bone_any(r,["Hips"]) and bone_any(r,["Head"])),None)
if lap_rig is None:
    raise RuntimeError("Lap humanoid armature not found")

LF=rig_frame(lap_rig)
LP=local_points(lap_body,LF,True)

zmin=0.02
neck_values=[z for z in [MF["z"]["neck"],LF["z"]["neck"]] if z is not None]
zmax=min(neck_values) if neck_values else 0.88
zmax=min(zmax,0.92)

angles={"front":0.0,"three_quarter":45.0,"side":90.0}
profiles={
    "Mona":{k:profile(MP,a,zmin,zmax) for k,a in angles.items()},
    "Lap":{k:profile(LP,a,zmin,zmax) for k,a in angles.items()}
}

landmarks={}
for key in ["hips","spine","chest","neck"]:
    vals=[v for v in [MF["z"].get(key),LF["z"].get(key)] if v is not None]
    if vals:
        landmarks[key]=sum(vals)/len(vals)

if "hips" in landmarks:
    landmarks["upper_thigh"]=max(zmin,landmarks["hips"]-0.10)
if "spine" in landmarks and "hips" in landmarks:
    landmarks["waist"]=(landmarks["spine"]+landmarks["hips"])*0.5

slabs={k:{"z":z,"Mona":slab(MP,z),"Lap":slab(LP,z)} for k,z in landmarks.items()}

def nearest_width(rows,z):
    if not rows:
        return None
    return min(rows,key=lambda r:abs(r["z"]-z))["width"]

zs=[zmin+(zmax-zmin)*i/120 for i in range(121)]
front_diff=[]
side_diff=[]
for z in zs:
    mw=nearest_width(profiles["Mona"]["front"],z)
    lw=nearest_width(profiles["Lap"]["front"],z)
    md=nearest_width(profiles["Mona"]["side"],z)
    ld=nearest_width(profiles["Lap"]["side"],z)
    if None not in (mw,lw):
        front_diff.append(abs(mw-lw))
    if None not in (md,ld):
        side_diff.append(abs(md-ld))

report={
    "ok":True,
    "stage":"R_Master_v9_Fix5_TrueBodyMeshCompare",
    "method":"exact Lap Body_Mesh + skeletal coordinate frames + arm-weight exclusion + no image renderer",
    "critical_audit":{
        "selected_lap_mesh":lap_body.name,
        "selected_lap_vertices":len(lap_body.data.vertices),
        "lap_meshes":audit,
        "previous_fix_invalid_reason":"Fix4 used largest-vertex heuristic and selected Face_Mesh."
    },
    "frames":{
        "Mona_z":MF["z"],
        "Lap_z":LF["z"],
        "compare_z":[zmin,zmax]
    },
    "profiles":profiles,
    "slabs":slabs,
    "metrics":{
        "mean_abs_front_width_diff_over_stature":sum(front_diff)/len(front_diff) if front_diff else None,
        "mean_abs_side_depth_diff_over_stature":sum(side_diff)/len(side_diff) if side_diff else None
    },
    "source_modified":False,
    "weights_transferred":False,
    "rest_pose_baked":False,
    "final_vrm":False
}

with open(os.path.join(out,"R_Master_v9_Fix5_true_body_profiles.json"),"w") as f:
    json.dump(report,f)

print("[v9fix5] TRUE_BODY_OK",lap_body.name,len(lap_body.data.vertices),report["metrics"],flush=True)
''',encoding="utf-8")

if REPORT.exists() and REPORT.stat().st_size>10000:
    try:
        old=json.loads(REPORT.read_text())
        if old.get("critical_audit",{}).get("selected_lap_mesh")!="Body_Mesh":
            REPORT.unlink()
    except:
        REPORT.unlink()

if not REPORT.exists():
    r=subprocess.run(
        [str(BLENDER),"--background","--python",str(SCRIPT),
         "--","--mona",str(MONA),"--lap",str(LAPFBX),"--out",str(OUT)],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True
    )
    print("\n".join([x for x in r.stdout.splitlines() if "[v9fix5]" in x or "Traceback" in x or "Error" in x])[-6000:])
    if r.returncode!=0:
        print(r.stdout[-12000:])
        raise RuntimeError("Fix5 extraction failed")

D=json.loads(REPORT.read_text())
print("✓ true body geometry extraction complete")
print(json.dumps(D["critical_audit"],ensure_ascii=False,indent=2))
print(json.dumps(D["metrics"],indent=2))
\n

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

D=json.loads(REPORT.read_text())
P=D["profiles"]
zlo,zhi=D["frames"]["compare_z"]

def outline(view,title,filename,zrange=None):
    lo,hi=(zrange if zrange else (zlo,zhi))
    fig,ax=plt.subplots(figsize=(5.6,8.0))
    for name,ls in [("Mona","-"),("Lap","--")]:
        rows=[r for r in P[name][view] if lo<=r["z"]<=hi]
        z=np.array([r["z"] for r in rows])
        mn=np.array([r["min"] for r in rows])
        mx=np.array([r["max"] for r in rows])
        ax.plot(mn,z,ls,label=name)
        ax.plot(mx,z,ls)
    ax.axvline(0,linewidth=.8)
    ax.grid(alpha=.2)
    ax.legend()
    ax.set_aspect("equal",adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("span / skeletal stature")
    ax.set_ylabel("height / skeletal stature")
    fig.tight_layout()
    p=OUT/filename
    fig.savefig(p,dpi=170)
    plt.close(fig)
    return p

sl=D["slabs"]

def region(keys,pad=.06):
    vals=[sl[k]["z"] for k in keys if k in sl]
    if not vals:
        return (zlo,zhi)
    return (max(zlo,min(vals)-pad),min(zhi,max(vals)+pad))

plots=[
    outline("front","TRUE BODY front envelope","R_Master_v9_Fix5_front.png"),
    outline("side","TRUE BODY side envelope","R_Master_v9_Fix5_side.png"),
    outline("three_quarter","TRUE BODY 45deg envelope","R_Master_v9_Fix5_three_quarter.png"),
    outline("front","Chest / waist compare","R_Master_v9_Fix5_chest_waist.png",region(["chest","spine","waist"],.05)),
    outline("front","Pelvis / upper-thigh compare","R_Master_v9_Fix5_pelvis_thigh.png",region(["hips","upper_thigh"],.07)),
    outline("front","Neck interface compare","R_Master_v9_Fix5_neck.png",region(["neck"],.045))
]

rows=[]
for k,v in sl.items():
    m=v.get("Mona") or {}
    l=v.get("Lap") or {}
    rows.append([k,v["z"],m.get("width"),l.get("width"),m.get("depth"),l.get("depth")])
rows=sorted(rows,key=lambda x:x[1],reverse=True)

fig,ax=plt.subplots(figsize=(8.5,3.2+0.3*len(rows)))
ax.axis("off")
tbl=ax.table(
    cellText=[
        [r[0],f"{r[1]:.3f}",
         f"{r[2]:.3f}" if r[2] is not None else "-",
         f"{r[3]:.3f}" if r[3] is not None else "-",
         f"{r[4]:.3f}" if r[4] is not None else "-",
         f"{r[5]:.3f}" if r[5] is not None else "-"]
        for r in rows
    ],
    colLabels=["landmark","z","Mona width","Lap width","Mona depth","Lap depth"],
    loc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1,1.35)
ax.set_title("True-body slab comparison")
fig.tight_layout()
tp=OUT/"R_Master_v9_Fix5_slabs.png"
fig.savefig(tp,dpi=170)
plt.close(fig)
plots.append(tp)

print("✓ plots ready")
\n

In [ ]:
from IPython.display import display,Image,Markdown
display(Markdown("## v9 Fix5 · TRUE Body Mesh Compare"))
display(Markdown("**关键校验：Lap 必须显示 Body_Mesh。不是 Body_Mesh 就不要采用结果。**"))

for p in plots:
    display(Markdown("### "+p.stem.replace("R_Master_v9_Fix5_","").replace("_"," ")))
    display(Image(filename=str(p),width=480))

ZIP=OUT/"R_Master_v9_Fix5_Review.zip"
if ZIP.exists():
    ZIP.unlink()

with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in plots:
        z.write(p,p.name)
    z.write(REPORT,REPORT.name)

print("✓ Review ZIP:",ZIP,round(ZIP.stat().st_size/1024/1024,2),"MiB")
from google.colab import files
files.download(str(ZIP))
\n